# Label Analysis — Data-Driven Tier Selection for NIID-Bench Partitioning

This notebook performs a **data-driven** comparison of label-proxy strategies for partitioning CircuitNet-N28 in a Non-IID federated learning setup.

## What this notebook answers

| Section | Question |
|---------|----------|
| A | What does the actual violation-rate distribution look like? |
| B | Are the proposed fixed cutoffs (0/2/8/20%) data-balanced? |
| C | What do data-driven (quantile) cutoffs give instead? |
| D | How strongly are tiers correlated with the 6 base designs? |
| E | Option D — does a multi-statistic proxy (rate + hotspot count) add value? |
| F | Option E — what natural tier skew does design-based partitioning produce? |
| G | Dirichlet parameter sweep — which β gives useful Non-IID severity? |
| H | Summary comparison table + data-driven recommendation |

## Setup note

The notebook works both when the real dataset is present (`../../drc_prediction/training_set/`) and when it is absent — in the latter case a statistically faithful mock is generated that reproduces the known heavy right-skew of DRC violation rates.

In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as mcm
from scipy import stats as scipy_stats
from scipy.stats import entropy as scipy_entropy

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.titlesize': 11,
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 0.8,
    'axes.facecolor': '#f5f5f5',
    'figure.facecolor': 'white',
})

sys.path.insert(0, os.path.abspath('.'))
from partitioning import LabelTierAssigner

VIOLATION_THRESHOLD = 0.1
FEATURE_DIR = '../../drc_prediction/training_set/feature/'
LABEL_DIR   = '../../drc_prediction/training_set/label/'
print('Imports OK.')

## Data Loading

Try to load real CircuitNet-N28 label files. Fall back to a statistically realistic mock if the dataset is absent.  
The mock reproduces the known property from FEATURES_DESCRIPTION §6.7: *most pixels are negative* — i.e. the per-sample violation rate is strongly right-skewed.

In [ ]:
def parse_sample_name(filename: str) -> dict:
    """Parse a CircuitNet-N28 filename into design-space components."""
    basename = filename.replace('.npy', '')
    parts = basename.split('-')
    if parts[0].isdigit():
        parts = parts[1:]
    if len(parts) >= 2 and len(parts[1]) == 1 and parts[1].isalpha():
        design_name = parts[0] + '-' + parts[1]
        rest = parts[2:]
    else:
        design_name = parts[0]
        rest = parts[1:]
    if len(rest) < 6:
        raise ValueError(f'Cannot parse: {filename}')
    return {
        'design_name':     design_name,
        'macro_count':     rest[0],
        'clock_ns':        float(rest[1][1:]),
        'utilization':     float(rest[2][1:]),
        'macro_placement': rest[3][1:],
        'power_mesh':      rest[4][1:],
        'filler_insertion':rest[5][1:],
        'filename':        filename,
    }


feature_exists = os.path.isdir(FEATURE_DIR)
label_exists   = os.path.isdir(LABEL_DIR)

if feature_exists and label_exists:
    files = [f for f in os.listdir(FEATURE_DIR) if f.endswith('.npy')]
    print(f'Real dataset: {len(files)} feature files found.')
    records = []
    for fname in files:
        try:
            records.append(parse_sample_name(fname))
        except Exception as e:
            print(f'  Skipping {fname}: {e}')
    df_meta = pd.DataFrame(records)

    print('Loading label .npy files...')
    rates = []
    for fname in df_meta['filename']:
        lpath = os.path.join(LABEL_DIR, fname)
        try:
            arr = np.load(lpath)
            rates.append(float(np.mean(arr >= VIOLATION_THRESHOLD)))
        except Exception:
            rates.append(float('nan'))
    df_meta['violation_rate'] = rates
    df_meta = df_meta.dropna(subset=['violation_rate']).reset_index(drop=True)
    DATASET_SOURCE = 'real'

else:
    print('Dataset not found — generating realistic mock (heavy right-skew).')
    # Realistic simulation: violation_rate is Beta(0.4, 4) rescaled to [0, 0.5].
    # This produces a strong concentration near 0 with a long tail, matching §6.7.
    rng = np.random.default_rng(42)
    N = 10242
    DESIGNS = ['RISCY-a', 'RISCY-b', 'RISCY-c', 'RISCY-d', 'RISCY-e', 'RISCY-f']
    CLOCKS  = [1.0, 1.5, 2.0, 2.5, 3.0, 3.3]
    UTILS   = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

    design_baseline = {
        # Each design has a slightly different baseline violation propensity
        'RISCY-a': 0.010, 'RISCY-b': 0.025, 'RISCY-c': 0.005,
        'RISCY-d': 0.040, 'RISCY-e': 0.015, 'RISCY-f': 0.030,
    }
    records = []
    violation_rates = []
    for _ in range(N):
        d  = rng.choice(DESIGNS)
        mc = rng.choice(['1','2','3','4'])
        c  = rng.choice(CLOCKS)
        u  = rng.choice(UTILS)
        mp = rng.choice(['1','2','3'])
        pm = rng.choice(['1','2'])
        fi = rng.choice(['0','1'])
        # Rate driven by utilization (main factor) + clock (secondary) + design baseline
        u_eff  = 0.18 * ((u - 0.4) / 0.5) ** 2     # non-linear: high util → much more violations
        c_eff  = 0.04 * (1.0 - c / 3.3)             # shorter clock → more timing violations
        base   = design_baseline[d] + u_eff + c_eff
        noise  = rng.beta(0.5, 5.0) * 0.12           # heavy right-skewed noise
        rate   = float(np.clip(base + noise, 0.0, 1.0))
        fname  = f'{d}-{mc}-c{c}-u{u}-m{mp}-p{pm}-f{fi}.npy'
        records.append({
            'design_name':     d,
            'macro_count':     mc,
            'clock_ns':        float(c),
            'utilization':     float(u),
            'macro_placement': mp,
            'power_mesh':      pm,
            'filler_insertion':fi,
            'filename':        fname,
        })
        violation_rates.append(rate)

    df_meta = pd.DataFrame(records)
    df_meta['violation_rate'] = violation_rates
    DATASET_SOURCE = 'mock'

print(f'\nSource: {DATASET_SOURCE}  |  Samples: {len(df_meta)}')
print(df_meta['violation_rate'].describe().to_string())

---
## Section A — Violation-Rate Distribution

Before choosing cutoffs, we must understand the empirical distribution of per-sample violation rates.

In [ ]:
vr = df_meta['violation_rate'].values

# Percentiles to report
pctls = [1, 5, 10, 25, 50, 75, 90, 95, 99]
pctl_vals = np.percentile(vr, pctls)
print('Percentiles of violation_rate:')
for p, v in zip(pctls, pctl_vals):
    print(f'  P{p:02d}: {v:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Section A — Violation-Rate Distribution', fontsize=13, fontweight='bold')

# (1) Linear histogram
ax = axes[0]
ax.hist(vr, bins=80, color='steelblue', alpha=0.8, edgecolor='white')
for bnd, col in zip([0.02, 0.08, 0.20], ['red','orange','green']):
    ax.axvline(bnd, color=col, linestyle='--', linewidth=1.4, label=f'Fixed cutoff {bnd}')
ax.set_title('Histogram (linear scale)')
ax.set_xlabel('violation_rate')
ax.set_ylabel('Sample count')
ax.legend(fontsize=8)

# (2) Log-scale Y to see the tails
ax = axes[1]
ax.hist(vr, bins=80, color='steelblue', alpha=0.8, edgecolor='white')
ax.set_yscale('log')
for bnd, col in zip([0.02, 0.08, 0.20], ['red','orange','green']):
    ax.axvline(bnd, color=col, linestyle='--', linewidth=1.4, label=f'Fixed cutoff {bnd}')
ax.set_title('Histogram (log Y scale — reveals tail)')
ax.set_xlabel('violation_rate')
ax.set_ylabel('Sample count (log)')
ax.legend(fontsize=8)

# (3) Empirical CDF
ax = axes[2]
sorted_vr = np.sort(vr)
cdf = np.arange(1, len(sorted_vr) + 1) / len(sorted_vr)
ax.plot(sorted_vr, cdf, color='steelblue', linewidth=1.5)
for bnd, col, lbl in zip(
    [0.02, 0.08, 0.20],
    ['red','orange','green'],
    ['Fixed 2%','Fixed 8%','Fixed 20%']
):
    pct_below = float(np.mean(vr < bnd)) * 100
    ax.axvline(bnd, color=col, linestyle='--', linewidth=1.4,
               label=f'{lbl} ({pct_below:.1f}% of samples)')
for q, qv in zip([25,50,75], np.percentile(vr, [25,50,75])):
    ax.axhline(q/100, color='purple', linestyle=':', linewidth=0.9,
               label=f'Q{q} = {qv:.4f}')
ax.set_title('Empirical CDF')
ax.set_xlabel('violation_rate')
ax.set_ylabel('Cumulative fraction')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

---
## Section B — Fixed Cutoffs Analysis (Proposed: 0 / 2% / 8% / 20%)

Apply the proposed tier boundaries and measure per-tier sample counts.  
A tier is considered **critically small** if it has fewer than 200 samples (< 2% of dataset) —  
too small for meaningful FL training with multiple parties.

In [ ]:
FIXED_BOUNDARIES = [0.0, 0.02, 0.08, 0.20, 1.01]
FIXED_NAMES = ['clean (0-2%)', 'low (2-8%)', 'medium (8-20%)', 'high (20-100%)']
MIN_VIABLE_SAMPLES = 200

assigner_fixed = LabelTierAssigner(threshold=VIOLATION_THRESHOLD, boundaries=FIXED_BOUNDARIES)
df_meta['tier_fixed'] = df_meta['violation_rate'].apply(assigner_fixed.assign_tier)

counts_fixed = df_meta['tier_fixed'].value_counts().sort_index()
pcts_fixed   = counts_fixed / len(df_meta) * 100

print('Fixed cutoffs tier distribution:')
for t, name in enumerate(FIXED_NAMES):
    cnt = counts_fixed.get(t, 0)
    pct = pcts_fixed.get(t, 0)
    flag = '  *** TOO SMALL ***' if cnt < MIN_VIABLE_SAMPLES else ''
    print(f'  Tier {t} [{name}]: {cnt:5d} samples ({pct:.1f}%){flag}')

print(f'\nBalance ratio (min/max tier count): {counts_fixed.min()/counts_fixed.max():.3f}')
print(f'Imbalance factor (max/min):         {counts_fixed.max()/counts_fixed.min():.1f}x')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Section B — Fixed Cutoffs (0/2%/8%/20%)', fontsize=13, fontweight='bold')

colors_fixed = ['#4caf50', '#ff9800', '#f44336', '#9c27b0']

ax = axes[0]
bars = ax.bar(FIXED_NAMES, [counts_fixed.get(t,0) for t in range(4)],
              color=colors_fixed, alpha=0.85, edgecolor='white')
ax.axhline(MIN_VIABLE_SAMPLES, color='black', linestyle='--', linewidth=1.2,
           label=f'Min viable ({MIN_VIABLE_SAMPLES})')
ax.set_title('Sample count per tier')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=20)
ax.legend(fontsize=9)
for bar, cnt in zip(bars, [counts_fixed.get(t,0) for t in range(4)]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
            f'{cnt}', ha='center', fontsize=9)

ax = axes[1]
ax.hist(vr, bins=100, color='steelblue', alpha=0.7, edgecolor='white')
for bnd, col, name in zip(FIXED_BOUNDARIES[1:-1], ['red','orange','green'], FIXED_NAMES[1:]):
    ax.axvline(bnd, color=col, linestyle='--', linewidth=1.5, label=f'{bnd}')
ax.set_title('Violation-rate histogram with fixed boundaries')
ax.set_xlabel('violation_rate')
ax.set_ylabel('Count')
ax.legend(fontsize=9, title='cutoff')

plt.tight_layout()
plt.show()

---
## Section C — Data-Driven Cutoffs: Quartile-Based vs Fixed

Alternative A from the review: place boundaries at the 25th, 50th, 75th percentiles of `violation_rate`,  
guaranteeing exactly equal tier sizes. We also test a **hybrid** approach: force the first boundary at 0.02  
(preserving the semantically meaningful clean/non-clean split) and use tertiles of the remaining samples.

In [ ]:
# --- A1: pure quartile cutoffs ---
q25, q50, q75 = np.percentile(vr, [25, 50, 75])
QUARTILE_BOUNDARIES = [0.0, float(q25), float(q50), float(q75), 1.01]
print(f'Quartile boundaries: {[f"{b:.4f}" for b in QUARTILE_BOUNDARIES]}')

assigner_qrt = LabelTierAssigner(threshold=VIOLATION_THRESHOLD, boundaries=QUARTILE_BOUNDARIES)
df_meta['tier_quartile'] = df_meta['violation_rate'].apply(assigner_qrt.assign_tier)

# --- A2: hybrid — keep 0.02 as first split, then tertiles of the rest ---
vr_above_clean = vr[vr >= 0.02]
h_33, h_67 = np.percentile(vr_above_clean, [33, 67])
HYBRID_BOUNDARIES = [0.0, 0.02, float(h_33), float(h_67), 1.01]
print(f'Hybrid boundaries:   {[f"{b:.4f}" for b in HYBRID_BOUNDARIES]}')

assigner_hyb = LabelTierAssigner(threshold=VIOLATION_THRESHOLD, boundaries=HYBRID_BOUNDARIES)
df_meta['tier_hybrid'] = df_meta['violation_rate'].apply(assigner_hyb.assign_tier)

# ---- Comparison table ----
schemes = [
    ('Fixed (0/2/8/20%)', 'tier_fixed', FIXED_BOUNDARIES, FIXED_NAMES),
    ('Quartile Q25/50/75', 'tier_quartile', QUARTILE_BOUNDARIES,
     [f'T{i}: [{QUARTILE_BOUNDARIES[i]:.3f},{QUARTILE_BOUNDARIES[i+1]:.3f})' for i in range(4)]),
    ('Hybrid (2%+tertiles)', 'tier_hybrid', HYBRID_BOUNDARIES,
     [f'T{i}: [{HYBRID_BOUNDARIES[i]:.3f},{HYBRID_BOUNDARIES[i+1]:.3f})' for i in range(4)]),
]

rows = []
for scheme_name, col, bnds, tier_labels in schemes:
    cnts = df_meta[col].value_counts().sort_index()
    balance = cnts.min() / cnts.max()
    imbalance = cnts.max() / cnts.min()
    rows.append({
        'Scheme': scheme_name,
        'Tier 0': int(cnts.get(0,0)),
        'Tier 1': int(cnts.get(1,0)),
        'Tier 2': int(cnts.get(2,0)),
        'Tier 3': int(cnts.get(3,0)),
        'Min tier': int(cnts.min()),
        'Max/Min ratio': round(float(imbalance), 1),
        'Balance (min/max)': round(float(balance), 3),
    })

cmp_df = pd.DataFrame(rows).set_index('Scheme')
print('\nCutoff scheme comparison:')
print(cmp_df.to_string())

# ---- Visualization ----
fig, axes = plt.subplots(1, 3, figsize=(21, 5))
fig.suptitle('Section C — Cutoff Scheme Comparison', fontsize=13, fontweight='bold')

palette = ['#4caf50', '#ff9800', '#f44336', '#9c27b0']

for ax, (scheme_name, col, bnds, tier_labels) in zip(axes, schemes):
    cnts = df_meta[col].value_counts().sort_index()
    bars = ax.bar(range(4), [cnts.get(t,0) for t in range(4)],
                  color=palette, alpha=0.85, edgecolor='white')
    ax.set_xticks(range(4))
    ax.set_xticklabels([f'T{t}' for t in range(4)])
    ax.set_title(scheme_name)
    ax.set_ylabel('Sample count')
    ax.axhline(MIN_VIABLE_SAMPLES, color='black', linestyle='--',
               linewidth=1.0, label='Min viable')
    ax.legend(fontsize=8)
    for bar, cnt in zip(bars, [cnts.get(t,0) for t in range(4)]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                f'{cnt}', ha='center', fontsize=9)
    # Add boundary annotation
    ax.text(0.98, 0.98,
            'Boundaries:\n' + '\n'.join([f'{b:.3f}' for b in bnds[1:-1]]),
            transform=ax.transAxes, fontsize=7, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.show()

---
## Section D — Design–Tier Correlation

The dataset is generated from **6 base RTL designs**. If tier assignment is strongly correlated with design identity, a "label skew" partition will in fact create a **feature skew** partition — relevant to declare explicitly in the report.

We measure:
- Box plots of `violation_rate` per design
- Cramér's V (association between design and tier — 0 = independent, 1 = perfect association)
- Heatmap of tier proportion per design

In [ ]:
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    """Cramér's V: strength of association between two categorical variables."""
    ct = pd.crosstab(x, y)
    chi2, _, _, _ = chi2_contingency(ct)
    n = ct.values.sum()
    k = min(ct.shape) - 1
    return float(np.sqrt(chi2 / (n * k))) if k > 0 else 0.0


designs = sorted(df_meta['design_name'].unique())
tier_col = 'tier_fixed'

# Cramér's V for all three schemes
print('Cramér\'s V (design_name ↔ tier) — 0=independent, 1=perfect:')
for scheme_name, col, *_ in schemes:
    v = cramers_v(df_meta['design_name'], df_meta[col])
    print(f'  {scheme_name:<30s}: V = {v:.4f}')

print()

# Kruskal-Wallis: does violation_rate differ significantly across designs?
groups = [df_meta.loc[df_meta['design_name']==d, 'violation_rate'].values for d in designs]
stat, pval = scipy_stats.kruskal(*groups)
print(f'Kruskal-Wallis test (violation_rate across designs):')
print(f'  H = {stat:.2f},  p = {pval:.2e}  (p<0.05 → designs differ significantly in violation rate)')

# Per-design stats
print('\nViolation rate per design:')
print(df_meta.groupby('design_name')['violation_rate']
      .agg(['mean','median','std','min','max'])
      .round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 5))
fig.suptitle('Section D — Design–Tier Correlation', fontsize=13, fontweight='bold')

# (1) Box plots per design
ax = axes[0]
cmap_d = mcm.get_cmap('Set2', len(designs))
data_per_design = [df_meta.loc[df_meta['design_name']==d, 'violation_rate'].values for d in designs]
bp = ax.boxplot(data_per_design, patch_artist=True, notch=False,
                medianprops={'color':'black','linewidth':1.5},
                flierprops={'marker':'.','markersize':2,'alpha':0.3})
for patch, i in zip(bp['boxes'], range(len(designs))):
    patch.set_facecolor(cmap_d(i))
    patch.set_alpha(0.8)
for bnd, col in zip(FIXED_BOUNDARIES[1:-1], ['red','orange','green']):
    ax.axhline(bnd, color=col, linestyle='--', linewidth=1.0, alpha=0.7)
ax.set_xticks(range(1, len(designs)+1))
ax.set_xticklabels(designs, rotation=30, ha='right', fontsize=9)
ax.set_title('Violation rate per design (fixed boundaries shown)')
ax.set_ylabel('violation_rate')

# (2) Heatmap: tier proportion per design (fixed cutoffs)
ax = axes[1]
n_tiers = 4
mat = np.zeros((len(designs), n_tiers))
for i, d in enumerate(designs):
    sub = df_meta.loc[df_meta['design_name']==d, tier_col]
    for t in range(n_tiers):
        mat[i, t] = (sub == t).sum() / len(sub)

im = ax.imshow(mat, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
for r in range(len(designs)):
    for c in range(n_tiers):
        val = mat[r, c]
        ax.text(c, r, f'{val:.2f}', ha='center', va='center',
                fontsize=8, color='white' if val > 0.55 else 'black')
ax.set_xticks(range(n_tiers))
ax.set_xticklabels(FIXED_NAMES, rotation=25, ha='right', fontsize=8)
ax.set_yticks(range(len(designs)))
ax.set_yticklabels(designs, fontsize=9)
ax.set_title('Tier proportion per design (fixed cutoffs)')
ax.set_xlabel('Tier')
plt.colorbar(im, ax=ax, fraction=0.04, label='Fraction')

# (3) Same heatmap with quartile cutoffs
ax = axes[2]
mat_q = np.zeros((len(designs), n_tiers))
for i, d in enumerate(designs):
    sub = df_meta.loc[df_meta['design_name']==d, 'tier_quartile']
    for t in range(n_tiers):
        mat_q[i, t] = (sub == t).sum() / len(sub)

im2 = ax.imshow(mat_q, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
for r in range(len(designs)):
    for c in range(n_tiers):
        val = mat_q[r, c]
        ax.text(c, r, f'{val:.2f}', ha='center', va='center',
                fontsize=8, color='white' if val > 0.55 else 'black')
ax.set_xticks(range(n_tiers))
ax.set_xticklabels([f'Q{i}' for i in range(4)], fontsize=9)
ax.set_yticks(range(len(designs)))
ax.set_yticklabels(designs, fontsize=9)
ax.set_title('Tier proportion per design (quartile cutoffs)')
ax.set_xlabel('Tier')
plt.colorbar(im2, ax=ax, fraction=0.04, label='Fraction')

plt.tight_layout()
plt.show()

---
## Section E — Option D: Multi-Statistic Proxy (Rate + Hotspot Count)

Instead of using only the fraction of violated pixels, we add **hotspot cluster count** — the number of distinct connected components with at least one pixel ≥ threshold.  
This distinguishes *few concentrated violations* from *many scattered violations*, which RouteNet processes differently.

We evaluate:
1. Whether hotspot count adds variance beyond violation_rate alone (Pearson r)
2. Whether a 2D clustering in (rate, count) space reveals structure absent in 1D

> **Note**: For real data, hotspot count is computed directly from label .npy arrays.  
> For mock data, it is simulated as correlated with violation_rate (high rate → more clusters).

In [ ]:
if DATASET_SOURCE == 'real':
    from scipy.ndimage import label as nd_label
    print('Computing hotspot cluster counts from label .npy files...')
    hotspot_counts = []
    for fname in df_meta['filename']:
        lpath = os.path.join(LABEL_DIR, fname)
        try:
            arr = np.load(lpath)
            binary = (arr >= VIOLATION_THRESHOLD).astype(int)
            _, n_clusters = nd_label(binary)
            hotspot_counts.append(float(n_clusters))
        except Exception:
            hotspot_counts.append(float('nan'))
    df_meta['hotspot_count'] = hotspot_counts
else:
    # Simulate: hotspot_count ~ Poisson(lambda = rate * 256*256 / avg_cluster_size)
    # avg_cluster_size ≈ 10 tiles (small but realistic)
    rng_hs = np.random.default_rng(99)
    # Number of clusters is roughly proportional to rate, but with saturation and noise
    # Use a Poisson model: E[n_clusters] = clip(rate * 200, 1, 500)
    lam = np.clip(df_meta['violation_rate'].values * 200, 0.5, 500.0)
    df_meta['hotspot_count'] = rng_hs.poisson(lam).astype(float)
    print('Simulated hotspot_count added (Poisson model).')

df_meta = df_meta.dropna(subset=['hotspot_count'])

# --- Pearson correlation between violation_rate and hotspot_count ---
r, pval = scipy_stats.pearsonr(df_meta['violation_rate'], df_meta['hotspot_count'])
print(f'\nPearson r(violation_rate, hotspot_count) = {r:.4f}  (p = {pval:.2e})')
print('  r > 0.95 → hotspot count mostly redundant with violation rate')
print('  r < 0.70 → hotspot count adds substantial independent information')

# Spearman (rank-based, more robust for skewed distributions)
r_s, pval_s = scipy_stats.spearmanr(df_meta['violation_rate'], df_meta['hotspot_count'])
print(f'Spearman r(violation_rate, hotspot_count) = {r_s:.4f}  (p = {pval_s:.2e})')

# --- 2D scatter: violation_rate vs hotspot_count ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Section E — Multi-Statistic Proxy: Rate + Hotspot Count',
             fontsize=13, fontweight='bold')

ax = axes[0]
n_plot = min(3000, len(df_meta))
sample = df_meta.sample(n_plot, random_state=0)
cmap_t = mcm.get_cmap('tab10', 4)
for t in range(4):
    sub = sample[sample['tier_fixed'] == t]
    ax.scatter(sub['violation_rate'], sub['hotspot_count'],
               s=6, alpha=0.4, color=cmap_t(t), label=FIXED_NAMES[t])
ax.set_xlabel('violation_rate')
ax.set_ylabel('hotspot_count')
ax.set_title(f'2D proxy space (Pearson r = {r:.3f})')
ax.legend(fontsize=8, markerscale=3)

# Residuals: hotspot_count after regressing out violation_rate
ax = axes[1]
X = df_meta['violation_rate'].values
Y = df_meta['hotspot_count'].values
coef = np.polyfit(X, Y, 1)
residuals = Y - np.polyval(coef, X)
ax.hist(residuals, bins=60, color='steelblue', alpha=0.8, edgecolor='white')
ax.set_title('Residuals of hotspot_count ~ violation_rate')
ax.set_xlabel('Residual (unexplained variance in hotspot count)')
ax.set_ylabel('Count')
ax.text(0.02, 0.97, f'Residual std: {residuals.std():.2f}\nMean: {residuals.mean():.2f}',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

# Interpretation
if abs(r) > 0.90:
    print('\n→ HIGH correlation: hotspot_count is largely redundant. Stick with violation_rate alone.')
elif abs(r) > 0.70:
    print('\n→ MODERATE correlation: hotspot_count adds some information. 2D proxy may help.')
else:
    print('\n→ LOW correlation: hotspot_count is a meaningful independent axis. 2D proxy recommended.')

---
## Section F — Option E: Natural Skew from Design-Based Partitioning

Instead of forcing tier skew, we partition by the **6 base RTL designs** — each party = one design family.  
We then *measure* what label (tier) skew naturally emerges.  
This represents the most realistic federated EDA scenario (different chip companies = different designs).

In [ ]:
design_partitions = [df_meta[df_meta['design_name'] == d].reset_index(drop=True)
                     for d in designs]

print('Design-based partition sizes:')
for d, p in zip(designs, design_partitions):
    tier_dist = p['tier_fixed'].value_counts().sort_index()
    tier_str = ', '.join([f'T{t}:{tier_dist.get(t,0)}' for t in range(4)])
    print(f'  {d}: {len(p):5d} samples  [{tier_str}]')

# Entropy of tier distribution per design partition
global_tier_counts = df_meta['tier_fixed'].value_counts().sort_index().values
global_entropy = scipy_entropy(global_tier_counts)
print(f'\nGlobal tier entropy: {global_entropy:.4f} nats')
print('Per-design tier entropy:')
for d, p in zip(designs, design_partitions):
    cnts = p['tier_fixed'].value_counts().reindex(range(4), fill_value=0).values
    ent = scipy_entropy(cnts)
    print(f'  {d}: entropy = {ent:.4f}  (Δ from global: {ent - global_entropy:+.4f})')

# Compare label entropy MAD for design-based vs synthetic partitioners
design_entropies = []
for p in design_partitions:
    cnts = p['tier_fixed'].value_counts().reindex(range(4), fill_value=0).values
    design_entropies.append(scipy_entropy(cnts))
design_entropy_mad = float(np.mean(np.abs(np.array(design_entropies) - global_entropy)))
print(f'\nLabel entropy MAD (design-based): {design_entropy_mad:.4f}')
print('  (Compare with Dirichlet strategies in partitioning_visualization.ipynb)')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Section F — Design-Based Partitioning (natural skew)',
             fontsize=13, fontweight='bold')

palette_t = ['#4caf50','#ff9800','#f44336','#9c27b0']

ax = axes[0]
bottom = np.zeros(len(designs))
for t in range(4):
    vals = np.array([p['tier_fixed'].value_counts().get(t, 0) for p in design_partitions])
    ax.bar(designs, vals, bottom=bottom, color=palette_t[t],
           label=FIXED_NAMES[t], alpha=0.85)
    bottom += vals
ax.set_title('Tier distribution per design (stacked)')
ax.set_ylabel('Sample count')
ax.tick_params(axis='x', rotation=25)
ax.legend(title='Tier (fixed)', fontsize=8)

ax = axes[1]
# Normalized (proportion)
bottom = np.zeros(len(designs))
for t in range(4):
    vals = np.array([
        (p['tier_fixed'] == t).sum() / len(p) for p in design_partitions
    ])
    ax.bar(designs, vals, bottom=bottom, color=palette_t[t],
           label=FIXED_NAMES[t], alpha=0.85)
    bottom += vals
ax.set_title('Tier proportion per design (normalized)')
ax.set_ylabel('Proportion')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=25)
ax.legend(title='Tier (fixed)', fontsize=8)

plt.tight_layout()
plt.show()

---
## Section G — Dirichlet Parameter Sweep (Option B)

Given our tier boundaries, how does the Dirichlet concentration parameter β control Non-IID severity?  
We sweep β ∈ {0.05, 0.1, 0.3, 0.5, 1.0, 5.0} and measure:
- Mean and std of per-party tier entropy
- Minimum party size (proxy for training data starvation risk)

In [ ]:
from partitioning import DirichletLabelPartitioner

N_PARTIES = 6  # match the 6 designs for comparability
ALPHAS = [0.05, 0.1, 0.3, 0.5, 1.0, 5.0]

# Use whichever tier scheme has better balance (compare fixed vs quartile)
# We test both below
results = []

for tier_col_g, scheme_label in [('tier_fixed', 'Fixed cutoffs'), ('tier_quartile', 'Quartile cutoffs')]:
    # Ensure tier column is in df
    df_for_dir = df_meta.copy()
    df_for_dir['tier'] = df_for_dir[tier_col_g]
    
    for alpha in ALPHAS:
        try:
            parts = DirichletLabelPartitioner(
                n_partitions=N_PARTIES, alpha=alpha, label_col='tier', seed=42
            ).partition(df_for_dir)
            sizes = np.array([len(p) for p in parts])
            entropies = []
            for p in parts:
                cnts = p['tier'].value_counts().reindex(range(4), fill_value=0).values
                entropies.append(scipy_entropy(cnts))
            entropies = np.array(entropies)
            results.append({
                'Scheme': scheme_label,
                'α (beta)': alpha,
                'Min size': int(sizes.min()),
                'Max size': int(sizes.max()),
                'Size std': round(float(sizes.std()), 0),
                'Mean tier entropy': round(float(entropies.mean()), 4),
                'Entropy std': round(float(entropies.std()), 4),
                'Entropy MAD': round(float(np.mean(np.abs(entropies - global_entropy))), 4),
            })
        except Exception as e:
            results.append({'Scheme': scheme_label, 'α (beta)': alpha, 'Error': str(e)})

dir_df = pd.DataFrame(results)
print('Dirichlet sweep results:')
print(dir_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 5))
fig.suptitle('Section G — Dirichlet Parameter Sweep', fontsize=13, fontweight='bold')

scheme_styles = {
    'Fixed cutoffs':    {'color': 'steelblue',  'marker': 'o'},
    'Quartile cutoffs': {'color': 'darkorange', 'marker': 's'},
}

for scheme_label, style in scheme_styles.items():
    sub = dir_df[dir_df['Scheme'] == scheme_label].dropna(subset=['Min size'])
    alphas_plot = sub['α (beta)'].values

    axes[0].plot(alphas_plot, sub['Min size'].values,
                 label=scheme_label, **style)
    axes[1].plot(alphas_plot, sub['Entropy MAD'].values,
                 label=scheme_label, **style)
    axes[2].plot(alphas_plot, sub['Size std'].values,
                 label=scheme_label, **style)

min_samples_per_party = max(50, int(len(df_meta) / N_PARTIES / 10))
axes[0].axhline(min_samples_per_party, color='red', linestyle='--',
                linewidth=1, label=f'Min viable ({min_samples_per_party})')
axes[0].set_xscale('log')
axes[0].set_title('Min party size vs α')
axes[0].set_xlabel('α (log scale)')
axes[0].set_ylabel('Min samples in any party')
axes[0].legend(fontsize=8)

axes[1].set_xscale('log')
axes[1].set_title('Label entropy MAD vs α\n(higher = more Non-IID)')
axes[1].set_xlabel('α (log scale)')
axes[1].set_ylabel('Entropy MAD from global')
axes[1].legend(fontsize=8)

axes[2].set_xscale('log')
axes[2].set_title('Partition size std vs α\n(higher = more quantity skew)')
axes[2].set_xlabel('α (log scale)')
axes[2].set_ylabel('Std of partition sizes')
axes[2].legend(fontsize=8)

for ax in axes:
    ax.set_xticks(ALPHAS)
    ax.set_xticklabels([str(a) for a in ALPHAS], fontsize=8)

plt.tight_layout()
plt.show()

# Recommended alpha range
print('\nRecommended α range for meaningful Non-IID experiments:')
for scheme_label in scheme_styles:
    sub = dir_df[dir_df['Scheme'] == scheme_label].dropna(subset=['Min size'])
    viable = sub[sub['Min size'] >= min_samples_per_party]
    if len(viable) > 0:
        alpha_min_viable = viable['α (beta)'].min()
        alpha_max_noniid = viable.loc[viable['Entropy MAD'] == viable['Entropy MAD'].max(), 'α (beta)'].values[0]
        print(f'  {scheme_label}: minimum viable α = {alpha_min_viable}  '
              f'| most Non-IID viable α = {alpha_max_noniid}')

---
## Section H — Summary: Data-Driven Comparison and Recommendation

Collects all metrics computed above into a single comparison table.

In [ ]:
# ---- Final comparison table ----

from partitioning import QuantityLabelPartitioner, IIDPartitioner

def eval_scheme(df_in, tier_col_name, n_parties=6, alphas=(0.1, 0.5), seed=42):
    """Return a dict of quality metrics for a given tier column."""
    df_e = df_in.copy()
    df_e['tier'] = df_e[tier_col_name]
    cnts = df_e['tier'].value_counts().sort_index()
    global_ent = scipy_entropy(cnts.values)

    results = {'tier_balance': round(float(cnts.min()/cnts.max()), 3),
               'min_tier_count': int(cnts.min()),
               'global_entropy': round(float(global_ent), 4)}

    # Dirichlet at two alpha values
    for alpha in alphas:
        try:
            parts = DirichletLabelPartitioner(
                n_partitions=n_parties, alpha=alpha, label_col='tier', seed=seed
            ).partition(df_e)
            sizes = np.array([len(p) for p in parts])
            ents  = np.array([
                scipy_entropy(p['tier'].value_counts().reindex(range(4), fill_value=0).values)
                for p in parts
            ])
            results[f'dir_α{alpha}_min_size'] = int(sizes.min())
            results[f'dir_α{alpha}_entropy_mad'] = round(float(np.mean(np.abs(ents - global_ent))), 4)
        except Exception as e:
            results[f'dir_α{alpha}_min_size'] = -1
            results[f'dir_α{alpha}_entropy_mad'] = float('nan')
    return results


rows_h = []
for scheme_label, tier_col_name in [
    ('Fixed cutoffs (0/2/8/20%)', 'tier_fixed'),
    ('Quartile Q25/50/75',         'tier_quartile'),
    ('Hybrid (2%+tertiles)',        'tier_hybrid'),
]:
    row = {'Labeling scheme': scheme_label}
    row.update(eval_scheme(df_meta, tier_col_name))
    rows_h.append(row)

final_df = pd.DataFrame(rows_h).set_index('Labeling scheme')
print('=== Final comparison table ===')
print(final_df.to_string())

In [ ]:
# Cramér's V and design correlation for all schemes
print('\nDesign ↔ Tier association (Cramér\'s V):')
for scheme_label, tier_col_name in [
    ('Fixed cutoffs', 'tier_fixed'),
    ('Quartile',      'tier_quartile'),
    ('Hybrid',        'tier_hybrid'),
]:
    v = cramers_v(df_meta['design_name'], df_meta[tier_col_name])
    interpretation = (
        'weak (good: label skew ≠ design skew)' if v < 0.1 else
        'moderate (label skew partially mirrors design)' if v < 0.3 else
        'strong (label skew ≈ design skew — declare explicitly)'
    )
    print(f'  {scheme_label:<22s}: V = {v:.4f}  → {interpretation}')

# Hotspot proxy added value
print(f'\nHotspot count Pearson r with violation_rate: {r:.4f}')
if abs(r) > 0.90:
    proxy_verdict = 'REDUNDANT — single-statistic violation_rate is sufficient.'
elif abs(r) > 0.70:
    proxy_verdict = 'PARTIALLY INFORMATIVE — adding hotspot count gives marginal gain.'
else:
    proxy_verdict = 'INFORMATIVE — 2D proxy (rate + count) recommended.'
print(f'Multi-statistic proxy verdict: {proxy_verdict}')

# Design-based partition natural entropy MAD
print(f'\nDesign-based partition label entropy MAD: {design_entropy_mad:.4f}')

In [ ]:
# Final scatter: all schemes on size-skew vs label-skew axes
from partitioning import QuantitySkewPartitioner

def entropy_mad_from_parts(parts, tier_col='tier_fixed', n_tiers=4):
    all_tiers = pd.concat(parts)[tier_col].value_counts().reindex(range(n_tiers), fill_value=0).values
    glob_ent = scipy_entropy(all_tiers)
    ents = [scipy_entropy(p[tier_col].value_counts().reindex(range(n_tiers), fill_value=0).values)
            for p in parts]
    return float(np.mean(np.abs(np.array(ents) - glob_ent)))

summary_pts = []

for tier_col_s, scheme_short in [('tier_fixed','Fixed'), ('tier_quartile','Quartile')]:
    df_s = df_meta.copy()
    df_s['tier'] = df_s[tier_col_s]

    for alpha in [0.1, 0.5, 5.0]:
        try:
            parts = DirichletLabelPartitioner(N_PARTIES, alpha=alpha, label_col='tier', seed=42).partition(df_s)
            sizes = np.array([len(p) for p in parts])
            summary_pts.append({
                'label': f'Dir-{scheme_short} α={alpha}',
                'size_std': float(sizes.std()),
                'label_entropy_mad': entropy_mad_from_parts(parts, tier_col_s),
                'min_size': int(sizes.min()),
                'color': 'steelblue' if scheme_short=='Fixed' else 'darkorange',
                'marker': 'o' if scheme_short=='Fixed' else 's',
            })
        except Exception:
            pass

    for k in [1, 2, 3]:
        try:
            parts = QuantityLabelPartitioner(N_PARTIES, n_labels_per_party=k, label_col='tier', seed=42).partition(df_s)
            sizes = np.array([len(p) for p in parts])
            summary_pts.append({
                'label': f'Qty-{scheme_short} k={k}',
                'size_std': float(sizes.std()),
                'label_entropy_mad': entropy_mad_from_parts(parts, tier_col_s),
                'min_size': int(sizes.min()),
                'color': 'steelblue' if scheme_short=='Fixed' else 'darkorange',
                'marker': '^' if scheme_short=='Fixed' else 'D',
            })
        except Exception:
            pass

# Design-based (reference)
design_sizes = np.array([len(p) for p in design_partitions])
summary_pts.append({
    'label': 'Design-based',
    'size_std': float(design_sizes.std()),
    'label_entropy_mad': design_entropy_mad,
    'min_size': int(design_sizes.min()),
    'color': 'crimson',
    'marker': '*',
})

fig, ax = plt.subplots(figsize=(12, 7))
for pt in summary_pts:
    ax.scatter(pt['size_std'], pt['label_entropy_mad'],
               s=120 if pt['marker']=='*' else 80,
               color=pt['color'], marker=pt['marker'],
               zorder=5, alpha=0.85)
    ax.annotate(pt['label'], (pt['size_std'], pt['label_entropy_mad']),
                textcoords='offset points', xytext=(5,3), fontsize=7)

ax.set_xlabel('Partition size std  (→ more quantity skew)')
ax.set_ylabel('Label entropy MAD  (→ more label skew)')
ax.set_title('All strategies: quantity skew vs label skew\n'
             'Blue=Fixed cutoffs  Orange=Quartile cutoffs  Red=Design-based')
plt.tight_layout()
plt.show()

print('\nFull summary:')
for pt in sorted(summary_pts, key=lambda x: -x['label_entropy_mad']):
    print(f"  {pt['label']:<30s}  size_std={pt['size_std']:6.0f}  "
          f"entropy_mad={pt['label_entropy_mad']:.4f}  min_size={pt['min_size']:5d}")

In [ ]:
# ---- Data-driven recommendation ----

fixed_cnts = df_meta['tier_fixed'].value_counts().sort_index()
qrt_cnts   = df_meta['tier_quartile'].value_counts().sort_index()
hyb_cnts   = df_meta['tier_hybrid'].value_counts().sort_index()

fixed_balance = fixed_cnts.min() / fixed_cnts.max()
qrt_balance   = qrt_cnts.min()   / qrt_cnts.max()
hyb_balance   = hyb_cnts.min()   / hyb_cnts.max()

print('=' * 70)
print('DATA-DRIVEN RECOMMENDATION')
print('=' * 70)
print()
print(f'Fixed cutoffs balance (min/max): {fixed_balance:.3f}')
print(f'Quartile cutoffs balance:        {qrt_balance:.3f}  (always 0.25)')
print(f'Hybrid cutoffs balance:          {hyb_balance:.3f}')
print()

if fixed_balance > 0.10:
    print('→ Fixed cutoffs are ACCEPTABLE (min tier > 10% of max tier).')
    print('  Proposed boundaries are data-supported; proceed with fixed cutoffs.')
    print('  Recommendation: use Fixed cutoffs for semantic interpretability.')
else:
    print('→ Fixed cutoffs produce a HEAVILY IMBALANCED tier (<10% balance).')
    print('  Recommendation: switch to Quartile or Hybrid cutoffs.')

print()
print('Proxy strategy:')
if abs(r) > 0.90:
    print('→ violation_rate alone is sufficient as the 1D label proxy.')
    print('  Option D (multi-statistic) does NOT add meaningful information.')
else:
    print('→ hotspot_count adds independent information. Option D is worth exploring.')

print()
print('Design correlation:')
v_fixed = cramers_v(df_meta['design_name'], df_meta['tier_fixed'])
if v_fixed > 0.3:
    print(f'→ V={v_fixed:.3f}: STRONG design↔tier correlation.')
    print('  DECLARE EXPLICITLY in report: tier-based label skew partially mirrors design skew.')
    print('  Consider design-based partitioning as a complementary experiment.')
elif v_fixed > 0.1:
    print(f'→ V={v_fixed:.3f}: MODERATE correlation. Mention in methodology; not critical.')
else:
    print(f'→ V={v_fixed:.3f}: WEAK correlation. Label skew is largely independent of design identity.')

print()
print('Partitioning method:')
print('→ For NIID-Bench compatibility:')
print('   • Quantity-based (#C=k): use k=1 or k=2 for strong skew, k=3 for mild.')
print('   • Dirichlet-based (β): use β=0.1-0.3 for strong, β=0.5-1.0 for moderate.')
print('→ Design-based partitioning: add as a realistic FL-EDA reference scenario.')
print()
print('=' * 70)